<a href="https://colab.research.google.com/github/viniciusdev772/IA/blob/main/treino_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GemmaMicro — Treino PT-BR no Colab

**Ordem de execução:** rode as células de cima para baixo.

- GPU recomendada: T4 (gratuita) ou A100 (Colab Pro)
- Salva automaticamente no Google Drive
- Tempo estimado: ~2h (T4) / ~40min (A100)

## 1. Verifica GPU e instala dependências

In [1]:
import subprocess, sys

# Checa GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NENHUMA — vá em Runtime > Change runtime type > T4 GPU')

# Instala dependências
!pip install -q torch tokenizers datasets trafilatura requests
print('Dependências instaladas.')

GPU: Tesla T4, 15360 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.3 MB/s eta 0:00:00
Dependências instaladas.


## 2. Monta Google Drive (salva modelo aqui)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Testa escrita
_t = f'{DRIVE_DIR}/_test.txt'
open(_t, 'w').write('ok')
assert os.path.exists(_t)
os.remove(_t)
print(f'Drive OK → {DRIVE_DIR}')

Mounted at /content/drive
Drive OK → /content/drive/MyDrive/GemmaMicro


## 3. Clona o repositório

In [3]:
import os

REPO = '/content/IA'
if not os.path.exists(REPO):
    !git clone https://github.com/viniciusdev772/IA.git {REPO}
else:
    !git -C {REPO} pull

print('Repo pronto.')

Cloning into '/content/IA'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 70 (delta 26), reused 59 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 615.88 KiB | 5.92 MiB/s, done.
Resolving deltas: 100% (26/26), done.
Repo pronto.


## 4. Coleta dados via crawler PT-BR

Crawla Wikipedia PT, G1, Agência Brasil, Folha, BBC Brasil, Nexo, etc.  
Sem downloads gigantes — coleta diretamente da web e salva parágrafos limpos.  
**Tempo estimado: 10-20 min** (2000 páginas, 16 threads paralelas).

In [4]:
import sys, os, shutil
sys.path.insert(0, '/content/IA')

REPO      = '/content/IA'
DATA_DIR  = '/content/datasets'
DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'
CRAWL_OUT = f'{DATA_DIR}/dataset_crawl.txt'

os.makedirs(DATA_DIR, exist_ok=True)

# ── Reutiliza crawl do Drive se já existir ─────────────────────────────────
drive_crawl = f'{DRIVE_DIR}/dataset_crawl.txt'
if not os.path.exists(CRAWL_OUT) and os.path.exists(drive_crawl):
    shutil.copy(drive_crawl, CRAWL_OUT)
    print(f'Crawl reutilizado do Drive: {os.path.getsize(CRAWL_OUT)/1024/1024:.1f} MB')

# ── Roda crawler se necessário ─────────────────────────────────────────────
if not os.path.exists(CRAWL_OUT):
    print('Iniciando crawler PT-BR (3000 páginas)...')

    import importlib

    # exec_module executa o código do módulo (redefine OUTPUT/MAX_PAGES/WORKERS),
    # então as sobreescritas devem vir DEPOIS de exec_module
    spec = importlib.util.spec_from_file_location('crawl', f'{REPO}/crawl.py')
    crawl_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(crawl_mod)

    # Sobrescreve após exec_module para não ser resetado
    crawl_mod.OUTPUT    = CRAWL_OUT
    crawl_mod.MAX_PAGES = 3000
    crawl_mod.WORKERS   = 16

    crawl_mod.crawl()

    if not os.path.exists(CRAWL_OUT):
        raise FileNotFoundError(f'Crawler terminou mas {CRAWL_OUT} não foi criado — verifique crawl.py')

    print(f'\nCrawl concluído: {os.path.getsize(CRAWL_OUT)/1024/1024:.1f} MB')

    # ── Limpa e deduplica via clean_crawl ─────────────────────────────────
    spec2 = importlib.util.spec_from_file_location('clean_crawl', f'{REPO}/clean_crawl.py')
    clean_mod = importlib.util.module_from_spec(spec2)
    spec2.loader.exec_module(clean_mod)
    clean_mod.INPUT  = CRAWL_OUT
    clean_mod.OUTPUT = CRAWL_OUT
    clean_mod.main()

    print(f'Dataset limpo: {os.path.getsize(CRAWL_OUT)/1024/1024:.1f} MB')

    # Salva cópia no Drive para reuso futuro
    shutil.copy(CRAWL_OUT, drive_crawl)
    print(f'Backup no Drive: {drive_crawl}')
else:
    print(f'dataset_crawl.txt já existe ({os.path.getsize(CRAWL_OUT)/1024/1024:.1f} MB), pulando crawler.')


Crawl reutilizado do Drive: 11.3 MB
dataset_crawl.txt já existe (11.3 MB), pulando crawler.


## 5. Lista todos os datasets disponíveis

In [5]:
import os, glob

REPO = '/content/IA'
DATA_DIR = '/content/datasets'

# Datasets do repo (pequenos, já commitados)
repo_datasets = [
    f'{REPO}/dataset_portugues_br.txt',
    f'{REPO}/frases_treinamento.txt',
    f'{REPO}/dataset_v2.txt',
    f'{REPO}/dataset_v3.txt',
    f'{REPO}/dataset_wiki_pt.txt',
]

# Datasets baixados
downloaded = glob.glob(f'{DATA_DIR}/*.txt')

all_paths = [p for p in repo_datasets + downloaded if os.path.exists(p)]

print(f'Datasets encontrados ({len(all_paths)}):')
total_mb = 0
for p in all_paths:
    mb = os.path.getsize(p) / 1024 / 1024
    total_mb += mb
    print(f'  {os.path.basename(p):40s} {mb:7.1f} MB')
print(f'  {"TOTAL":40s} {total_mb:7.1f} MB')

Datasets encontrados (6):
  dataset_portugues_br.txt                     0.0 MB
  frases_treinamento.txt                       0.0 MB
  dataset_v2.txt                               0.0 MB
  dataset_v3.txt                               0.0 MB
  dataset_wiki_pt.txt                          0.3 MB
  dataset_crawl.txt                           11.3 MB
  TOTAL                                       11.6 MB


## 6. Treina o modelo GemmaMicro

Parâmetros ajustáveis:
- `epochs` — mais épocas = mais tempo, melhor qualidade
- `ctx_len` — contexto. 256 usa menos VRAM que 512.
- `batch_size` — reduza para 16 se der OOM

In [ ]:
import sys, os, glob
sys.path.insert(0, '/content/IA')
import nn

REPO      = '/content/IA'
DATA_DIR  = '/content/datasets'
DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'

# Reconstrói all_paths (caso célula 5 não tenha rodado)
repo_datasets = [
    f'{REPO}/dataset_portugues_br.txt',
    f'{REPO}/frases_treinamento.txt',
    f'{REPO}/dataset_v2.txt',
    f'{REPO}/dataset_v3.txt',
    f'{REPO}/dataset_wiki_pt.txt',
]
downloaded = glob.glob(f'{DATA_DIR}/*.txt')
all_paths = [p for p in repo_datasets + downloaded if os.path.exists(p)]

print(f'Datasets para treino ({len(all_paths)}):')
for p in all_paths:
    mb = os.path.getsize(p) / 1024 / 1024
    print(f'  {os.path.basename(p):40s} {mb:7.1f} MB')

nn.train(
    data_paths=all_paths,
    save_dir=DRIVE_DIR,
    epochs=50,              # mais épocas = PPL mais baixo com dataset pequeno
    batch_size=32,          # suba para 64 se A100
    lr=6e-4,
    ctx_len=256,
    accum_steps=4,          # batch efetivo = 32*4 = 128
    d_model=256,
    n_heads=8,
    vocab_size=2000,        # 4000→2000: distribuição menor = PPL cai mais rápido
    max_lines_per_file=500_000,
    log_every=20,
)


Datasets para treino (6):
  dataset_portugues_br.txt                     0.0 MB
  frases_treinamento.txt                       0.0 MB
  dataset_v2.txt                               0.0 MB
  dataset_v3.txt                               0.0 MB
  dataset_wiki_pt.txt                          0.3 MB
  dataset_crawl.txt                           11.3 MB
Device: cuda | AMP: True | Grad accum: 4 (effective batch=128)
Carregando textos:
  /content/IA/dataset_portugues_br.txt: 156 linhas (cap=500000)
  /content/IA/frases_treinamento.txt: 438 linhas (cap=500000)
  /content/IA/dataset_v2.txt: 196 linhas (cap=500000)
  /content/IA/dataset_v3.txt: 297 linhas (cap=500000)
  /content/IA/dataset_wiki_pt.txt: 1,687 linhas (cap=500000)
  /content/datasets/dataset_crawl.txt: 89,579 linhas (cap=500000)
Total frases: 92353
Treinando BPE tokenizer:
  BPE (Rust): 92353 textos, vocab_size=2000, merges=1741
  BPE: vocab final 2000 (1741 merges)
Vocab size: 2000
  tokenizado 5,000/92,353
  tokenizado 10,000/92,3

## 7. Testa inferência

In [ ]:
import torch, sys
sys.path.insert(0, '/content/IA')
from nn import GemmaMicro, BPETokenizer

DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'

tokenizer = BPETokenizer.load(f'{DRIVE_DIR}/tokenizer.json')
ckpt = torch.load(f'{DRIVE_DIR}/model.pt', map_location='cpu', weights_only=True)
model = GemmaMicro(vocab_size=tokenizer.vocab_size)
model.load_state_dict(ckpt)
model.eval()
print(f'Modelo: {sum(p.numel() for p in model.parameters()):,} parâmetros')

def gerar(prompt, max_tokens=120, temp=0.8, top_k=50):
    ids = tokenizer.encode(prompt, add_special=False)
    ids = torch.tensor([ids])
    with torch.no_grad():
        for _ in range(max_tokens):
            logits = model(ids[:, -512:])
            logit = logits[0, -1, :] / temp
            v, _ = torch.topk(logit, top_k)
            logit[logit < v[-1]] = float('-inf')
            next_id = torch.multinomial(torch.softmax(logit, -1), 1)
            ids = torch.cat([ids, next_id.unsqueeze(0)], dim=1)
            if next_id.item() == 258:  # <eos>
                break
    return tokenizer.decode(ids[0].tolist(), skip_special=False)

prompts = [
    'a capital do brasil é',
    'a inteligência artificial',
    'o brasil é um país',
    'a língua portuguesa',
]
for p in prompts:
    print(f'\n→ {p}')
    print(gerar(p))

## 8. (Opcional) Baixa modelo do Drive para /content

In [ ]:
import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'
LOCAL_DIR = '/content/gemma_micro'
os.makedirs(LOCAL_DIR, exist_ok=True)

for f in ['model.pt', 'tokenizer.json']:
    src = f'{DRIVE_DIR}/{f}'
    dst = f'{LOCAL_DIR}/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        mb = os.path.getsize(dst)/1024/1024
        print(f'Copiado {f}: {mb:.1f} MB')

print('Pronto. Use load_model(\'gemma_micro\') localmente.')